# Insectes : classification vs détection vs pose — où le modèle regarde-t-il ?Quatre modèles, une seule tâche : attribuer l'image à l'un des groupes.| modèle | supervision | dataset | peut s'abstenir ? ||---|---|---|---|| `cls` | label global | `AllSpecies-cls` | non || `cls_bg` | label global + classe `background` | `AllSpecies-cls-bg` | oui (prédit `background`) || `detect` | boîtes | `AllSpecies-detect` | oui (aucune boîte) || `pose` | boîtes + keypoints | `AllSpecies-pose` | oui (aucune boîte) |`cls` et `cls_bg` forment la comparaison **avant / après** l'ajout de la classe `background`,sur les métriques comme sur l'attention.À lancer après `fuze_datasets.py` puis `create_background_class.py`.### MétriquesUne seule chose est mesurée : **la performance par classe**, sous deux formes.- **Matrice de confusion**, avec une colonne `abstention` supplémentaire. L'abstention n'est  donc pas une métrique séparée : elle est visible là où elle se produit, classe par classe.  La diagonale normalisée par ligne donne l'**accuracy par classe** (= rappel).- **F1 one-vs-rest par classe**, où l'abstention compte comme faux négatif.### Deux z-scores, deux questions| | question | dénominateur ||---|---|---|| `z_intra` | ce modèle est-il relativement faible sur cette classe ? | écart-type des F1 **entre classes** || `z_diff` | l'écart avant/après est-il réel ou du bruit ? | écart-type **bootstrap** de la différence |`z_intra` est celui décrit littéralement dans une approche one-vs-rest, mais avec 3 ou 4classes son dénominateur repose sur 3 ou 4 points : il ordonne les classes, il ne teste rien.`z_diff` est le seul défendable pour comparer deux modèles, parce qu'il estime la variance parrééchantillonnage des images de test.

## 1. ConfigurationSeule cellule à éditer.

In [ ]:
from pathlib import Pathimport numpy as np, pandas as pd, cv2, torch, yamlimport matplotlib.pyplot as pltfrom ultralytics import YOLOROOT = Path("./models/datasets")# nom -> (tache ultralytics, dataset)MODEL_SPECS = {    "cls":    ("classify", ROOT / "AllSpecies-cls"),                     # sans background    "cls_bg": ("classify", ROOT / "AllSpecies-cls-bg"),                  # avec background    "detect": ("detect",   ROOT / "AllSpecies-detect" / "yolo-config.yaml"),    "pose":   ("pose",     ROOT / "AllSpecies-pose"   / "yolo-config.yaml"),}NAMES = list(MODEL_SPECS)BEFORE, AFTER = "cls", "cls_bg"          # couple compare avant/apresCF_DIR      = ROOT / "counterfactual"CF_VARIANTS = ["mean_noise", "telea", "gray"]CF_TRAINED  = "mean_noise"               # variante vue par cls_bg a l'entrainementBACKGROUND  = "background"IMGSZ, EPOCHS, BATCH, SCALE, SEED = 640, 100, 16, "n", 0DEVICE = 0 if torch.cuda.is_available() else "cpu"SPLIT       = "test"N_PER_GROUP = 10        # images expliquees par groupeCONF        = 0.10      # seuil bas : on veut voir l'abstention, pas la masquerOCC_PATCH, OCC_STRIDE = 96, 48TARGET_LAYER = None     # None -> dernier bloc C2PSA du backboneN_BOOTSTRAP  = 2000     # pour z_difftorch.manual_seed(SEED); np.random.seed(SEED)RUNS = Path("runs"); RUNS.mkdir(exist_ok=True)POSE_CFG = yaml.safe_load(open(MODEL_SPECS["pose"][1]))_n = POSE_CFG["names"]GROUPS = [_n[i] for i in sorted(_n)] if isinstance(_n, dict) else list(_n)NC, CHANCE = len(GROUPS), 1.0 / len(GROUPS)ABSTAIN = -1print("device:", DEVICE, "| groupes:", GROUPS)

## 2. Letterbox, masque, splitLe letterbox est **identique** à celui écrit sur disque par `fuze_datasets.py` pour lesdatasets `cls`, et à celui qu'Ultralytics applique en interne pour `detect`/`pose`. C'est cetteidentité qui rend les cartes de saillance superposables entre les quatre modèles.

In [ ]:
def letterbox(img, size=IMGSZ):    h, w = img.shape[:2]    r = min(size / h, size / w)    nh, nw = max(1, int(round(h * r))), max(1, int(round(w * r)))    canvas = np.full((size, size, 3), 114, np.uint8)    px, py = (size - nw) // 2, (size - nh) // 2    canvas[py:py + nh, px:px + nw] = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)    return canvas, r, px, pydef load_square(path, size=IMGSZ):    img = cv2.imread(str(path))    if img is None:        raise FileNotFoundError(path)    if img.shape[0] == size and img.shape[1] == size:        return img                    # deja letterboxee (dataset cls, contre-factuel)    return letterbox(img, size)[0]def bbox_mask(label_path, image_path, size=IMGSZ):    """Masque insecte dans le repere letterboxe."""    h, w = cv2.imread(str(image_path)).shape[:2]    r = min(size / h, size / w)    nh, nw = max(1, int(round(h * r))), max(1, int(round(w * r)))    px, py = (size - nw) // 2, (size - nh) // 2    m = np.zeros((size, size), np.uint8)    for line in Path(label_path).read_text().strip().splitlines():        p = line.split()        if len(p) < 5:            continue        cx, cy, bw, bh = map(float, p[1:5])        x1, y1 = int((cx - bw / 2) * w * r) + px, int((cy - bh / 2) * h * r) + py        x2, y2 = int((cx + bw / 2) * w * r) + px, int((cy + bh / 2) * h * r) + py        m[max(0, y1):max(0, y2), max(0, x1):max(0, x2)] = 255    return mdef norm01(a):    a = np.nan_to_num(a.astype(np.float32))    lo, hi = a.min(), a.max()    return np.zeros_like(a) if hi - lo < 1e-12 else (a - lo) / (hi - lo)def load_split(split=SPLIT):    root  = Path(POSE_CFG["path"])    imdir = root / POSE_CFG.get(split, f"images/{split}")    rows = []    for img in sorted(imdir.rglob("*")):        if img.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".webp"}:            continue        lbl = Path(str(img).replace("/images/", "/labels/")).with_suffix(".txt")        if not lbl.exists() or not lbl.read_text().strip():            continue        cid = int(float(lbl.read_text().split()[0]))        rows.append({"image": str(img), "label": str(lbl),                     "class_id": cid, "group": GROUPS[cid]})    return pd.DataFrame(rows)test_df = load_split()print(f"{len(test_df)} images dans le split '{SPLIT}'")print(test_df.group.value_counts().to_string())

## 3. Entraînement des quatre modèlesHyperparamètres appariés. `cls` et `cls_bg` ne diffèrent que par leur dataset — c'est ce qui rend leur comparaison interprétable.

In [ ]:
BASE = {"classify": f"yolo26{SCALE}-cls.pt", "detect": f"yolo26{SCALE}.pt",        "pose": f"yolo26{SCALE}-pose.pt"}def train(name):    task, data = MODEL_SPECS[name]    out = RUNS / "train" / name / "weights" / "best.pt"    if out.exists():        print(f"{name}: deja entraine -> {out}")        return out    YOLO(BASE[task]).train(data=str(data), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,                           seed=SEED, device=DEVICE, project=str(RUNS / "train"),                           name=name, exist_ok=True, deterministic=True)    return outmodels = {n: YOLO(str(train(n))) for n in NAMES}for n, m in models.items():    print(f"{n:<7} classes: {m.names}")

## 4. Prédiction image-level unifiée`detect` / `pose` : classe de la boîte la plus confiante ; aucune boîte = abstention.`cls_bg` : classe la plus probable ; `background` = abstention.`cls` : classe la plus probable, jamais d'abstention.Le remappage passe par `model.names`, pas par l'ordre des dossiers : Ultralytics ordonne lesclasses de classification **alphabétiquement**. Avec des groupes en minuscules, `background`passerait en tête et décalerait tous les indices. Ce remappage rend le notebook insensible aupiège, et il vaut aussi pour `cls` et `cls_bg` qui n'ont pas le même nombre de classes.

In [ ]:
MAP = {}for n in NAMES:    task = MODEL_SPECS[n][0]    raw = models[n].names    raw = raw if isinstance(raw, dict) else dict(enumerate(raw))    name2idx = {v: k for k, v in raw.items()}    missing = [g for g in GROUPS if g not in name2idx]    assert not missing, f"{n} : groupes absents {missing} (vu : {list(name2idx)})"    MAP[n] = {"task": task,              "cols": [name2idx[g] for g in GROUPS],              "bg": name2idx.get(BACKGROUND)}    print(f"{n:<7} colonnes groupes {MAP[n]['cols']}  background -> {MAP[n]['bg']}")assert MAP[AFTER]["bg"] is not None, (    f"{AFTER} n'a pas de classe '{BACKGROUND}'. Lancer create_background_class.py "    "puis reentrainer.")assert MAP[BEFORE]["bg"] is None, f"{BEFORE} ne devrait pas avoir de classe '{BACKGROUND}'."def model_class_index(name, group_idx):    """Indice de la classe dans l'espace de sortie du modele (Grad-CAM en a besoin)."""    return MAP[name]["cols"][group_idx]def predict(name, images, chunk=32):    """-> scores (N, NC) par groupe, abstain (N,) booleen."""    info, model = MAP[name], models[name]    scores, abstain = [], []    for i in range(0, len(images), chunk):        kw = dict(imgsz=IMGSZ, verbose=False, device=DEVICE)        if info["task"] != "classify":            kw["conf"] = CONF        for r in model.predict(images[i:i + chunk], **kw):            if info["task"] == "classify":                p = r.probs.data.cpu().numpy().astype(np.float32)                scores.append(p[info["cols"]])                abstain.append(info["bg"] is not None and int(p.argmax()) == info["bg"])            else:                s = np.zeros(NC, np.float32)                empty = r.boxes is None or len(r.boxes) == 0                if not empty:                    c = r.boxes.cls.cpu().numpy().astype(int)                    f = r.boxes.conf.cpu().numpy()                    for k in range(NC):                        if (c == k).any():                            s[k] = f[c == k].max()                scores.append(s)                abstain.append(empty)    return np.stack(scores), np.array(abstain, bool)def labels(scores, abstain):    return np.where(abstain, ABSTAIN, scores.argmax(1))imgs   = [load_square(p) for p in test_df.image]y_true = test_df.class_id.to_numpy()PRED   = {}for n in NAMES:    sc, ab = predict(n, imgs)    PRED[n] = labels(sc, ab)    print(f"{n:<7} predictions calculees ({(PRED[n] == ABSTAIN).sum()} abstentions)")

## 5. Matrices de confusion et F1 one-vs-restLa colonne `abstention` fait partie de la matrice : un modèle qui refuse de répondre n'est nijuste ni faux au sens habituel, et le noyer dans un taux global effacerait *sur quelles classes*il refuse.La diagonale normalisée par ligne est l'**accuracy par classe**. Pour le F1 one-vs-rest,l'abstention compte comme faux négatif : ne pas répondre sur un vrai *Coleoptera* est unéchec de détection de cette classe.

In [ ]:
LABELS_CM = GROUPS + ["abstention"]def confusion(y, yp):    """(NC, NC+1) — lignes = vraie classe, derniere colonne = abstention."""    cm = np.zeros((NC, NC + 1), int)    for t, p in zip(y, yp):        cm[t, NC if p == ABSTAIN else p] += 1    return cmdef f1_one_vs_rest(y, yp):    """F1 par classe ; l'abstention compte comme faux negatif."""    out = np.zeros(NC)    for c in range(NC):        tp = int(((yp == c) & (y == c)).sum())        fp = int(((yp == c) & (y != c)).sum())        fn = int(((y == c) & (yp != c)).sum())     # inclut l'abstention        out[c] = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0    return outCM = {n: confusion(y_true, PRED[n]) for n in NAMES}F1 = {n: f1_one_vs_rest(y_true, PRED[n]) for n in NAMES}fig, axes = plt.subplots(1, len(NAMES), figsize=(4.2 * len(NAMES), 3.6))for ax, n in zip(np.atleast_1d(axes), NAMES):    cmn = CM[n] / np.maximum(CM[n].sum(1, keepdims=True), 1)    ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)    ax.set_xticks(range(NC + 1)); ax.set_xticklabels(LABELS_CM, rotation=45, ha="right", fontsize=7)    ax.set_yticks(range(NC));     ax.set_yticklabels(GROUPS, fontsize=7)    for i in range(NC):        for j in range(NC + 1):            ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=7,                    color="white" if cmn[i, j] > .5 else "black")    ax.set_title(n, fontsize=10)    ax.set_xlabel("predit", fontsize=8)axes[0].set_ylabel("vrai", fontsize=8)plt.tight_layout(); plt.show()acc = pd.DataFrame({n: np.diag(CM[n][:, :NC]) / np.maximum(CM[n].sum(1), 1) for n in NAMES},                   index=GROUPS)print("accuracy par classe (diagonale / effectif de la ligne)\n")print(acc.round(3).to_string())

### z-scores`z_intra` situe chaque classe **à l'intérieur d'un modèle** : positif = classe mieux reconnueque la moyenne des classes de ce modèle. Avec peu de classes son écart-type repose sur peu depoints, il ordonne mais ne teste rien.`z_diff` compare **deux modèles sur la même classe**. La variance vient d'un bootstrap appariésur les images de test : on rééchantillonne les mêmes indices pour les deux modèles, ce quiélimine la variabilité due au choix des images et ne laisse que l'écart entre modèles.`|z| > 2` ≈ écart difficilement attribuable au hasard d'échantillonnage. Un test réduit rendraces valeurs instables — regarder aussi l'intervalle.

In [ ]:
def z_intra(f1):    s = f1.std(ddof=0)    return (f1 - f1.mean()) / s if s > 1e-12 else np.zeros_like(f1)def z_diff(y, yp_a, yp_b, n_boot=N_BOOTSTRAP, seed=SEED):    """Bootstrap apparie : memes indices rééchantillonnes pour les deux modeles."""    rng = np.random.default_rng(seed)    n = len(y)    diffs = np.zeros((n_boot, NC))    for b in range(n_boot):        idx = rng.integers(0, n, n)        diffs[b] = f1_one_vs_rest(y[idx], yp_b[idx]) - f1_one_vs_rest(y[idx], yp_a[idx])    obs = f1_one_vs_rest(y, yp_b) - f1_one_vs_rest(y, yp_a)    sd = diffs.std(0, ddof=1)    z = np.where(sd > 1e-12, obs / np.maximum(sd, 1e-12), 0.0)    lo, hi = np.percentile(diffs, [2.5, 97.5], axis=0)    return obs, z, lo, hif1_tab = pd.DataFrame({n: F1[n] for n in NAMES}, index=GROUPS)zi_tab = pd.DataFrame({n: z_intra(F1[n]) for n in NAMES}, index=GROUPS)print("F1 one-vs-rest\n"); print(f1_tab.round(3).to_string())print("\nz_intra (par modele, entre classes)\n"); print(zi_tab.round(2).to_string())obs, z, lo, hi = z_diff(y_true, PRED[BEFORE], PRED[AFTER])comp = pd.DataFrame({f"F1_{BEFORE}": F1[BEFORE], f"F1_{AFTER}": F1[AFTER],                     "delta": obs, "z_diff": z, "IC95_bas": lo, "IC95_haut": hi},                    index=GROUPS)print(f"\ncomparaison {BEFORE} -> {AFTER} (ajout de la classe {BACKGROUND})\n")print(comp.round(3).to_string())for g, zz, d in zip(GROUPS, z, obs):    if abs(zz) > 2:        print(f"  {g:<14} {'gain' if d > 0 else 'perte'} significatif (z={zz:+.2f})")    else:        print(f"  {g:<14} ecart non distinguable du bruit (z={zz:+.2f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))f1_tab.plot(kind="bar", ax=axes[0], rot=0)axes[0].set_title("F1 one-vs-rest par classe", fontsize=10); axes[0].set_ylim(0, 1)axes[0].legend(fontsize=7)axes[1].bar(GROUPS, obs, yerr=[obs - lo, hi - obs], capsize=4, color="#4C72B0")axes[1].axhline(0, c="k", lw=1)axes[1].set_title(f"delta F1 : {AFTER} - {BEFORE}  (IC95 bootstrap)", fontsize=10)axes[1].tick_params(labelrotation=0)plt.tight_layout(); plt.show()

## 6. Grad-CAMDeux subtilités que masquerait un appel naïf à `predict()` :1. `predict()` s'exécute sous `inference_mode` : aucun gradient. On appelle donc le réseau   directement.2. Pour récupérer le logit de classe sans dépendre de la valeur de retour de `forward` (qui   change selon la tâche et la version), on pose un hook sur `head.cv3`.**Point critique.** Sur une tête end-to-end (YOLO26), la branche `one2one_cv3` opère sur desfeatures **détachées** : aucun gradient ne peut la traverser jusqu'au backbone. La CAM estdonc calculée sur la branche `one2many` (`cv3`), entraînée conjointement mais pas identique àcelle qui produit les prédictions. À mentionner en méthodologie.

In [ ]:
def get_net(name):    net = models[name].model.to(DEVICE if DEVICE != "cpu" else "cpu").float().eval()    for p in net.parameters():        p.requires_grad_(True)    return netdef get_layer(net):    if TARGET_LAYER is not None:        return net.model[TARGET_LAYER]    c2psa = [m for m in net.model if type(m).__name__.upper().startswith("C2PSA")]    assert c2psa, "Aucun bloc C2PSA : preciser TARGET_LAYER."    return c2psa[-1]          # dernier bloc du backbone = seul point commun aux 4 modelesdef gradcam(name, img, group_idx):    net, task = get_net(name), MAP[name]["task"]    head, layer = net.model[-1], get_layer(net)    cls_idx = model_class_index(name, group_idx)    store, cls_maps, handles = {}, [], []    def keep_act(m, i, o):            # ne rien renvoyer : un hook qui retourne une        store["a"] = o                # valeur remplacerait la sortie du module        if o.requires_grad:            o.register_hook(lambda g: store.__setitem__("g", g.detach()))    handles.append(layer.register_forward_hook(keep_act))    if task != "classify":        handles += [b.register_forward_hook(lambda m, i, o: cls_maps.append(o))                    for b in head.cv3]    x = torch.from_numpy(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)).permute(2, 0, 1)[None]    x = (x.float() / 255).to(next(net.parameters()).device)    prev = head.training    head.training = True              # sorties brutes : evite softmax / postprocess NMS-free    try:        out = net(x)        if task == "classify":            o = out[0] if isinstance(out, (list, tuple)) else out            score = o.reshape(-1)[cls_idx]        else:            score = torch.cat([m[0, cls_idx].reshape(-1) for m in cls_maps]).max()        net.zero_grad(set_to_none=True)        score.backward()    finally:        head.training = prev        for h in handles:            h.remove()    a, g = store["a"][0].detach(), store.get("g")    assert g is not None, "Aucun gradient : verifier TARGET_LAYER."    cam = (g[0].mean((1, 2), keepdim=True) * a).sum(0).clamp(min=0).cpu().numpy()    return norm01(cv2.resize(cam, (IMGSZ, IMGSZ)))

## 7. Vérification : la CAM est-elle class-specific ?Garde-fou minimal. Si les classes produisent la même carte, la couche cible est trop profonde pour discriminer et les heatmaps ne signifient rien. Tant que cette cellule n'affiche pas `OK` partout, ne pas interpréter la suite.

In [ ]:
probe = load_square(test_df.image.iloc[0])for n in NAMES:    maps = [gradcam(n, probe, c) for c in range(NC)]    ecart = np.mean([np.abs(maps[0] - m).mean() for m in maps[1:]]) if NC > 1 else 0.0    etat = "OK" if maps[0].std() > 1e-6 and ecart > 1e-4 else "PROBLEME"    print(f"{n:<7} ecart inter-classes {ecart:.5f}   [{etat}]")    if etat == "PROBLEME":        print(f"         -> essayer TARGET_LAYER parmi {list(get_net(n).model[-1].f)}")

## 8. OcclusionOn masque une fenêtre glissante et on mesure la chute du score de la vraie classe. Aucun gradient, aucune hypothèse d'architecture : **strictement la même procédure pour les quatre modèles**. C'est la méthode de référence quand elle contredit Grad-CAM.

In [ ]:
def occlusion(name, img, group_idx, patch=OCC_PATCH, stride=OCC_STRIDE):    base = predict(name, [img])[0][0, group_idx]    coords, variants = [], []    for yy in range(0, IMGSZ - patch + 1, stride):        for xx in range(0, IMGSZ - patch + 1, stride):            v = img.copy()            v[yy:yy + patch, xx:xx + patch] = 114     # meme gris que le letterbox            variants.append(v); coords.append((yy, xx))    drops = base - predict(name, variants)[0][:, group_idx]    acc = np.zeros((IMGSZ, IMGSZ), np.float32)    cnt = np.zeros((IMGSZ, IMGSZ), np.float32)    for (yy, xx), d in zip(coords, drops):        acc[yy:yy + patch, xx:xx + patch] += d        cnt[yy:yy + patch, xx:xx + patch] += 1    return norm01(np.maximum(acc / np.maximum(cnt, 1), 0))print(f"{(((IMGSZ - OCC_PATCH)//OCC_STRIDE)+1)**2} inferences par image et par modele")

## 9. Attention : localisation, et effet de la classe `background`- **EBPG** — fraction de la masse de saillance dans l'insecte. `1 - EBPG` = dépendance au fond.- **ratio** — EBPG normalisé par l'aire du masque. **C'est la colonne à lire** : un EBPG de  0.40 sur un insecte couvrant 40 % de l'image, c'est le hasard (ratio = 1).On explique toujours la **vraie** classe, pas la classe prédite : expliquer la classe préditemélangerait erreurs de classification et défauts de localisation.Le tableau final répond à la question qui motive la classe `background` : est-ce que forcer lemodèle à reconnaître le fond le pousse à regarder davantage l'insecte ?

In [ ]:
def ebpg(sal, mask):    tot = sal.sum()    return float(sal[mask > 0].sum() / tot) if tot > 0 else 0.0n_per = min(N_PER_GROUP, int(test_df.group.value_counts().min()))sample = test_df.groupby("group", group_keys=False).sample(n=n_per, random_state=SEED)print(f"{len(sample)} images expliquees ({n_per} par groupe)")rows, saliency = [], {}for n in NAMES:    for method, fn in (("gradcam", gradcam), ("occlusion", occlusion)):        for r in sample.itertuples():            img  = load_square(r.image)            mask = bbox_mask(r.label, r.image)            if mask.sum() == 0:                continue            sal = fn(n, img, r.class_id)            saliency[(n, method, r.image)] = sal            e, area = ebpg(sal, mask), float((mask > 0).mean())            rows.append({"model": n, "method": method, "group": r.group, "image": r.image,                         "ebpg": e, "aire": area, "ratio": e / area if area else np.nan})        print(f"{n}/{method} termine")sal_df = pd.DataFrame(rows)print("\nratio EBPG / aire  (1 = hasard)\n")print(sal_df.pivot_table(index="model", columns="method", values="ratio").round(2).to_string())

In [ ]:
# --- effet de la classe background sur l'attention, apparie image par image ---piv = sal_df.pivot_table(index=["method", "group", "image"], columns="model", values="ratio")piv = piv.dropna(subset=[BEFORE, AFTER])piv["delta"] = piv[AFTER] - piv[BEFORE]rng = np.random.default_rng(SEED)lines = []for method in sal_df.method.unique():    d = piv.xs(method, level="method")["delta"].to_numpy()    boot = np.array([rng.choice(d, len(d), replace=True).mean() for _ in range(N_BOOTSTRAP)])    lines.append({"method": method, "n": len(d), "delta_moyen": d.mean(),                  "z": d.mean() / boot.std(ddof=1) if boot.std() > 1e-12 else 0.0,                  "IC95_bas": np.percentile(boot, 2.5),                  "IC95_haut": np.percentile(boot, 97.5)})eff = pd.DataFrame(lines)print(f"effet de l'ajout de {BACKGROUND} sur le ratio de localisation "      f"({AFTER} - {BEFORE}, apparie par image)\n")print(eff.round(3).to_string(index=False))for r in eff.itertuples():    verdict = ("regarde DAVANTAGE l'insecte" if r.z > 2 else               "regarde MOINS l'insecte" if r.z < -2 else               "aucun effet distinguable du bruit")    print(f"  {r.method:<10} {verdict} (z={r.z:+.2f})")fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))sal_df.pivot_table(index="model", columns="method", values="ratio").plot(    kind="bar", ax=axes[0], rot=0)axes[0].axhline(1, ls="--", c="k", lw=1, label="hasard")axes[0].set_title("ratio EBPG / aire par modele", fontsize=10); axes[0].legend(fontsize=8)for method in sal_df.method.unique():    axes[1].hist(piv.xs(method, level="method")["delta"], bins=20, alpha=.55, label=method)axes[1].axvline(0, c="k", lw=1)axes[1].set_title(f"delta ratio par image : {AFTER} - {BEFORE}", fontsize=10)axes[1].legend(fontsize=8)plt.tight_layout(); plt.show()

## 10. Test contre-factuelLa mesure la plus solide du notebook : elle ne dépend d'aucune hypothèse d'explicabilité.Trois variantes produites par `create_background_class.py`, à lire **ensemble** :| variante | ce qu'elle fait | rôle ||---|---|---|| `mean_noise` | aplat + bruit + raccord flou | méthode d'entraînement de la classe `background` || `telea` | inpainting OpenCV, reconstruit le fond | **jamais vue à l'entraînement** || `gray` | aplat gris, aucun raccord | contrôle pur : réaction au trou |Si un modèle se comporte pareil sur `telea` et sur `gray`, il réagit à l'artefact, pas àl'absence d'insecte.

In [ ]:
def cf_index(variant):    d = CF_DIR / variant / SPLIT    return {p.stem: p for p in d.glob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg"}}idx   = {v: cf_index(v) for v in CF_VARIANTS}stems = [Path(p).stem for p in test_df.image]keep  = [i for i, s in enumerate(stems) if all(s in idx[v] for v in CF_VARIANTS)]print(f"{len(keep)}/{len(stems)} images appariees sur les {len(CF_VARIANTS)} variantes")assert keep, f"Aucune image trouvee dans {CF_DIR}. Lancer create_background_class.py."paired = test_df.iloc[keep].reset_index(drop=True)y_cf   = paired.class_id.to_numpy()cf_rows = []for n in NAMES:    for v in CF_VARIANTS:        sc, ab = predict(n, [load_square(idx[v][stems[i]]) for i in keep])        yp = labels(sc, ab)        cf_rows.append({"model": n, "variante": v,                        "encore_correct": float((yp == y_cf).mean()),                        "abstention": float(ab.mean())})cf_df = pd.DataFrame(cf_rows)fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))for ax, col, titre in zip(axes, ["encore_correct", "abstention"],                          ["classe encore correcte sans insecte", "taux d'abstention"]):    cf_df.pivot(index="model", columns="variante", values=col).plot(kind="bar", ax=ax, rot=0)    if col == "encore_correct":        ax.axhline(CHANCE, ls="--", c="k", lw=1, label=f"hasard ({CHANCE:.2f})")    ax.set_title(titre, fontsize=10); ax.legend(fontsize=7)plt.tight_layout(); plt.show()p_ok = cf_df.pivot(index="model", columns="variante", values="encore_correct")print("Lecture :\n")for n in NAMES:    real, ctrl = float(p_ok.loc[n, "telea"]), float(p_ok.loc[n, "gray"])    if abs(real - ctrl) < 0.05:        v = "identique au controle gris -> reagit au trou, pas a l'absence d'insecte. NON CONCLUANT."    elif real <= CHANCE + 0.05:        v = "retombe au niveau du hasard -> pas de dependance au fond detectable."    elif real < 0.5:        v = "dependance au fond moderee mais superieure au hasard."    else:        v = "dependance au fond FORTE."    print(f"  {n:<7} telea {real:.3f} / gray {ctrl:.3f} -> {v}")

## 11. Diagnostic : `cls_bg` a-t-il appris l'artefact ?Confond spécifique à la classe `background` : les images `mean_noise` sont dans**l'entraînement** de `cls_bg`. Il peut donc avoir appris la signature de l'artefact (bruituniforme, raccord flou) plutôt que l'absence d'insecte.Le test tient en une comparaison : si `cls_bg` s'abstient massivement sur `mean_noise` (vue)mais pas sur `telea` (jamais vue), il reconnaît l'artefact. `cls`, `detect` et `pose` n'ontjamais vu ces images : ils servent de témoins. Un écart chez eux est du bruit, un écart chez`cls_bg` seul est le symptôme.

In [ ]:
ab = cf_df.pivot(index="model", columns="variante", values="abstention")print(ab.round(3).to_string(), "\n")ecarts = {n: float(ab.loc[n, CF_TRAINED] - ab.loc[n, "telea"]) for n in NAMES}temoins = [ecarts[n] for n in NAMES if n != AFTER]print(f"ecart d'abstention ({CF_TRAINED} vue - telea jamais vue) :")for n in NAMES:    print(f"  {n:<7} {ecarts[n]:+.3f}" + ("   <- modele teste" if n == AFTER else "   (temoin)"))ref = float(np.mean(temoins))print(f"\nmoyenne des temoins : {ref:+.3f}")if ecarts[AFTER] - ref > 0.15:    print(f"\n=> {AFTER} s'abstient beaucoup plus sur la variante vue a l'entrainement.\n"          "   Il a appris la SIGNATURE DE L'ARTEFACT, pas l'absence d'insecte.\n"          "   Son abstention ne prouve donc pas qu'il regarde l'insecte, et la\n"          "   comparaison avant/apres des cellules 5 et 9 est a lire avec cette reserve.\n"          "   Correctif : entrainer background sur un MELANGE de methodes\n"          "   (mean_noise + telea + gray) et garder une 4e methode inedite pour le test.")else:    print(f"\n=> pas de sur-abstention specifique a la variante d'entrainement :\n"          f"   l'abstention de {AFTER} porte bien sur l'absence d'insecte.")

## 12. GalerieLigne = modèle, colonne = image. Comparer les lignes `cls` et `cls_bg` donne l'effet visuel de la classe `background` sur l'attention.

In [ ]:
def show(method, n=4):    picks = sample.sample(min(n, len(sample)), random_state=SEED)    fig, axes = plt.subplots(len(NAMES) + 1, len(picks),                             figsize=(3 * len(picks), 3 * (len(NAMES) + 1)), squeeze=False)    for j, r in enumerate(picks.itertuples()):        img, m = load_square(r.image), bbox_mask(r.label, r.image)        vis = img.copy()        x, yb, w, h = cv2.boundingRect((m > 0).astype(np.uint8))        cv2.rectangle(vis, (x, yb), (x + w, yb + h), (255, 255, 255), 2)        axes[0][j].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))        axes[0][j].set_title(r.group, fontsize=9)        if j == 0:            axes[0][j].set_ylabel("image", fontsize=10)        for i, name in enumerate(NAMES, start=1):            sal = saliency.get((name, method, r.image))            if sal is None:                continue            heat = cv2.applyColorMap((sal * 255).astype(np.uint8), cv2.COLORMAP_JET)            axes[i][j].imshow(cv2.cvtColor(cv2.addWeighted(heat, .45, img, .55, 0),                                           cv2.COLOR_BGR2RGB))            if j == 0:                axes[i][j].set_ylabel(name, fontsize=10)    for row in axes:        for a in row:            a.set_xticks([]); a.set_yticks([])    fig.suptitle(method, fontsize=12); plt.tight_layout(); plt.show()show("gradcam")show("occlusion")

## Comment lire les résultats1. **La colonne `abstention` de la matrice de confusion se lit avec la diagonale.** Un modèle   qui s'abstient beaucoup a une accuracy par classe faible sans être pour autant mauvais :   il refuse au lieu de se tromper. `cls` ne peut pas s'abstenir, sa colonne est vide par   construction — c'est la raison d'être de `cls_bg`.2. **`z_diff` avant `delta`.** Un écart de F1 de 0.04 sur 40 images de test n'est pas un   résultat. L'intervalle bootstrap dit s'il faut y croire.3. **`ratio` avant `ebpg`.** Un EBPG élevé sur des insectes qui remplissent l'image ne veut   rien dire. `ratio > 1` = concentration sur l'insecte supérieure au hasard.4. **Grad-CAM vs occlusion.** S'ils convergent, la conclusion tient. S'ils divergent,   l'occlusion l'emporte : Grad-CAM passe par la branche `one2many` et par les gradients, deux   sources d'artefact que l'occlusion n'a pas.5. **Le contre-factuel prime sur les cartes.** Une heatmap dit où le modèle *semble* regarder ;   le contre-factuel dit s'il *a besoin* de l'insecte.6. **La cellule 11 conditionne toute lecture de `cls_bg`.** Si elle signale l'apprentissage de   l'artefact, ni son abstention ni son gain d'attention ne prouvent quoi que ce soit.### Limites- Grad-CAM des modèles `detect`/`pose` : branche `one2many` (la branche d'inférence `one2one`  reçoit des features détachées, aucun gradient possible).- Masque = bbox → ratio optimiste en valeur absolue, mais comparable entre modèles.- `cls` et `cls_bg` n'ont pas le même nombre de classes : leurs F1 one-vs-rest restent  comparables car calculés sur les mêmes images à insecte et les mêmes classes cibles, mais  leurs probabilités de sortie ne sont pas calibrées de la même façon.- Le bootstrap suppose des images de test indépendantes. Si plusieurs images proviennent d'un  même spécimen ou d'une même séance de prise de vue, les intervalles sont trop étroits.